In [1]:
import batchalign as ba
from tqdm import tqdm
import argparse
import requests
from bs4 import BeautifulSoup
import re
from tqdm import tqdm
import os
import pandas as pd
import stanza
from pprint import pprint
import math
from dataclasses import dataclass
from typing import List, Dict, Any, Optional, Tuple
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor

In [ ]:
raw_train_news_df = pd.read_csv("F:/NT@B/Microsoft-sp26/MINDlarge_train/news.tsv", sep="\t")
raw_test_news_df = pd.read_csv("F:/NT@B/Microsoft-sp26/MINDlarge_test/news.tsv", sep="\t")

header_data = ["id", "category", "subcategory", "title", "abstract", "url", "title_entities", "abstract_entities"]
raw_train_news_df.columns = header_data
raw_test_news_df.columns = header_data

raw_train_news_df.head(), raw_test_news_df.head()

(       id category               subcategory  \
 0  N45436     news  newsscienceandtechnology   
 1  N23144   health                weightloss   
 2  N86255   health                   medical   
 3  N93187     news                 newsworld   
 4  N75236   health                    voices   
 
                                                title  \
 0    Walmart Slashes Prices on Last-Generation iPads   
 1                      50 Worst Habits For Belly Fat   
 2  Dispose of unwanted prescription drugs during ...   
 3  The Cost of Trump's Aid Freeze in the Trenches...   
 4  I Was An NBA Wife. Here's How It Affected My M...   
 
                                             abstract  \
 0  Apple's new iPad releases bring big deals on l...   
 1  These seemingly harmless habits are holding yo...   
 2                                                NaN   
 3  Lt. Ivan Molchanets peeked over a parapet of s...   
 4  I felt like I was a fraud, and being an NBA wi...   
 
                

In [8]:
train_news_df = raw_train_news_df[raw_train_news_df["category"] == "news"]
test_news_df = raw_test_news_df[raw_test_news_df["category"] == "news"]

train_news_df = train_news_df.reset_index(drop=True)
test_news_df = test_news_df.reset_index(drop=True)

train_news_df = train_news_df[~train_news_df["abstract"].isna()]
test_news_df = test_news_df[~test_news_df["abstract"].isna()]

test_news_df = test_news_df[~test_news_df["id"].isin(train_news_df["id"])]
test_news_df = test_news_df.reset_index(drop=True)

test_news_df

,id,category,subcategory,title,abstract,url,title_entities,abstract_entities
0,N26869,news,newsscienceandtechnology,The Cuisinart Griddler grill and panini press ...,That's a fantastic deal on one of my favorite ...,https://assets.msn.com/labs/mind/AAALSWr.html,"[{""Label"": ""Nonogram"", ""Type"": ""C"", ""WikidataI...",[]
1,N13021,news,newsscienceandtechnology,iPad Pro 12.9 (2018),"Apple rules the tablet roost, but is its newes...",https://assets.msn.com/labs/mind/AACkZc5.html,"[{""Label"": ""IPad Pro"", ""Type"": ""U"", ""WikidataI...","[{""Label"": ""IPad Pro"", ""Type"": ""U"", ""WikidataI..."
2,N52698,news,newsscienceandtechnology,Best universal remotes of 2019,"From Harmony to Caavo to, well, other Harmonys...",https://assets.msn.com/labs/mind/AACLWAP.html,[],"[{""Label"": ""Logitech Harmony"", ""Type"": ""U"", ""W..."
3,N82357,news,elections-2020-us,2020 Presidential Debates Fast Facts,Read CNN's Fast Facts on the 2020 presidential...,https://assets.msn.com/labs/mind/AAE2wMO.html,[],"[{""Label"": ""CNN"", ""Type"": ""M"", ""WikidataId"": ""..."
4,N35969,news,newsscienceandtechnology,Bernheim property avoided in preliminary I-65/...,Gov. Matt Bevin and the Kentucky Transportatio...,https://assets.msn.com/labs/mind/AAJfU9R.html,"[{""Label"": ""Interstate 65"", ""Type"": ""S"", ""Wiki...","[{""Label"": ""Kentucky Transportation Cabinet"", ..."
...,...,...,...,...,...,...,...,...
8622,N35700,news,newsus,Chicago water bill payments down $20 million t...,Chicago water bill payments are down $20 milli...,https://assets.msn.com/labs/mind/BBXc03a.html,"[{""Label"": ""Lori Lightfoot"", ""Type"": ""N"", ""Wik...","[{""Label"": ""Lori Lightfoot"", ""Type"": ""N"", ""Wik..."
8623,N129075,news,newsus,"Arlington man, 65, dies after he stumbled onto...",FORT WORTH -- A 65-year-old Arlington man has ...,https://assets.msn.com/labs/mind/BBXc084.html,[],"[{""Label"": ""Tarrant County, Texas"", ""Type"": ""G..."
8624,N87483,news,newsus,Passengers evacuate after California train col...,"The crash happened in Santa Fe Springs, Los An...",https://assets.msn.com/labs/mind/BBXc08G.html,"[{""Label"": ""California"", ""Type"": ""G"", ""Wikidat...","[{""Label"": ""Santa Fe Springs, California"", ""Ty..."
8625,N43703,news,newsus,Commuter train hits RV in fiery collision near...,An RV stopped on the tracks was hit by a commu...,https://assets.msn.com/labs/mind/BBXc08t.html,"[{""Label"": ""Los Angeles"", ""Type"": ""G"", ""Wikida...","[{""Label"": ""Los Angeles"", ""Type"": ""G"", ""Wikida..."


# CLAN Analysis

## Example Execution

In [56]:
doc = ba.Document.new("Hello, this is a transcript! I have two utterances.", 
                      media_path="audio.mp3", lang="eng")

# navigating the document
first_utterance = doc[0]
first_form = doc[0][0]
the_comma = doc[0][1]

assert the_comma.text == ','
assert the_comma.type == ba.TokenType.PUNCT

# taking a transcript
sentences = doc.transcript(include_tiers=True, strip=True)

In [57]:
sentences

['PAR: Hello, this is a transcript!', 'PAR: I have two utterances.']

## Mind Execution

In [37]:
CLAUSAL_DEPRELS = {"ccomp", "xcomp", "advcl", "acl", "acl:relcl", "parataxis"}
SENT_LABELS = {"S", "SBAR", "SBARQ", "SINV", "SQ"}
PUNCT_TAGS = {".", ",", ":", "``", "''", "-LRB-", "-RRB-", "#", "$"}

def clause_count_from_dependencies(stanza_sentence):
    count = 0

    # main clause: one root predicate
    root_words = [w for w in stanza_sentence.words if w.head == 0]
    if root_words:
        count += 1

    # embedded/subordinate clauses
    for w in stanza_sentence.words:
        if w.deprel in CLAUSAL_DEPRELS:
            count += 1

    return count

def clause_density(stanza_doc):
    densities = []
    for sent in stanza_doc.sentences:
        n_clauses = clause_count_from_dependencies(sent)
        densities.append(n_clauses)  # per sentence
    return densities

def dependency_lengths(stanza_sentence):
    dists = []
    for w in stanza_sentence.words:
        if w.head != 0:  # skip root
            dists.append(abs(w.id - w.head))
    return dists

def mean_dependency_length(stanza_doc):
    dist_per_sentence = []
    for sent in stanza_doc.sentences:
        dists = dependency_lengths(sent)
        dist_per_sentence.append(sum(dists) / len(dists) if dists else 0.0)
    return dist_per_sentence

def is_leaf(node) -> bool:
    return len(node.children) == 0

def is_preterminal(node) -> bool:
    return len(node.children) == 1 and is_leaf(node.children[0])

def is_punct_preterminal(node) -> bool:
    return is_preterminal(node) and node.label in PUNCT_TAGS

def leaf_text(node) -> str:
    return node.label

def iter_leaf_paths(node, path=None):
    """
    Yield tuples: (leaf_node, path)
    where path is a list of (parent_node, child_index) from root down to the leaf.
    """
    if path is None:
        path = []

    if is_leaf(node):
        yield node, path
        return

    for i, child in enumerate(node.children):
        yield from iter_leaf_paths(child, path + [(node, i)])

def iter_nodes(node):
    yield node
    for child in node.children:
        yield from iter_nodes(child)

def count_non_punct_words(tree) -> int:
    count = 0
    for node in iter_nodes(tree):
        if is_preterminal(node) and not is_punct_preterminal(node):
            count += 1
    return count

def left_embedding_depth(tree) -> int:
    """
    Constituency-based operationalization:
    For each leaf path, count how many ancestors keep the leaf on the left edge
    of an unfinished constituent, i.e. the child is the first child and the parent
    has material to the right.

    Sentence score = maximum such depth across leaves.
    """
    max_depth = 0

    for leaf, path in iter_leaf_paths(tree):
        depth = 0
        for parent, child_idx in path:
            if is_preterminal(parent):
                continue
            if child_idx == 0 and len(parent.children) > 1:
                depth += 1
        max_depth = max(max_depth, depth)

    return max_depth

def center_embedding_depth(tree) -> int:
    """
    Constituency-based operationalization:
    Count nested sentential constituents (S, SBAR, SBARQ, SINV, SQ)
    that occur in a non-final position of another sentential constituent.

    This captures center-embedding rather than simple right-branching.
    """
    def helper(node, active_center_depth=0):
        best = active_center_depth

        for i, child in enumerate(node.children):
            child_depth = active_center_depth

            if child.label in SENT_LABELS and not is_preterminal(child):
                # Embedded clause inside a non-final position -> center embedding increment
                if i < len(node.children) - 1:
                    child_depth += 1

            best = max(best, helper(child, child_depth))

        return best

    return helper(tree, 0)

def yngve_scores(tree) -> Dict[str, Any]:
    """
    For each leaf:
      score = sum over ancestors of the number of right siblings.
    Returns per-word scores plus sum/mean/max over words.
    """
    per_word = []

    for leaf, path in iter_leaf_paths(tree):
        # Ignore punctuation leaves by checking their preterminal parent
        if len(path) >= 1:
            parent, _ = path[-1]
            if is_punct_preterminal(parent):
                continue

        score = 0
        for parent, child_idx in path:
            score += (len(parent.children) - child_idx - 1)

        per_word.append({
            "word": leaf_text(leaf),
            "score": score
        })

    values = [x["score"] for x in per_word]
    return {
        "per_word": per_word,
        "sum": sum(values),
        "mean": (sum(values) / len(values)) if values else 0.0,
        "max": max(values) if values else 0
    }

def frazier_scores(tree) -> Dict[str, Any]:
    """
    A compact, reproducible Frazier-style implementation.

    For each leaf:
      - climb from the POS preterminal upward
      - only count nodes while the current node remains the leftmost child
        of its parent (once there is a left sibling, stop)
      - add:
          1.5 for sentence-level nodes: S, SBAR, SBARQ, SINV, SQ
          1.0 for other phrasal nonterminals
      - ignore the POS preterminal itself and punctuation

    This is a practical approximation; exact conventions vary across papers.
    """
    per_word = []

    for leaf, path in iter_leaf_paths(tree):
        if not path:
            continue

        preterminal, _ = path[-1]
        if is_punct_preterminal(preterminal):
            continue

        score = 0.0

        # path is [(root, idx), ..., (preterminal, idx_of_leaf)]
        # We score ancestors from the preterminal's parent upward.
        # Stop once the current node is not the leftmost child of its parent.
        # Current "node" starts as the preterminal.
        for level in range(len(path) - 2, -1, -1):
            parent, child_idx = path[level]

            # Count this parent node
            if parent.label in SENT_LABELS:
                score += 1.5
            else:
                score += 1.0

            # If this parent itself is not leftmost in its own parent, stop
            if level > 0:
                _, parent_idx_in_grandparent = path[level - 1]
                if parent_idx_in_grandparent != 0:
                    break

        per_word.append({
            "word": leaf_text(leaf),
            "score": score
        })

    values = [x["score"] for x in per_word]
    return {
        "per_word": per_word,
        "sum": sum(values),
        "mean": (sum(values) / len(values)) if values else 0.0,
        "max": max(values) if values else 0.0
    }

def analyze_doc(doc, surprisal_model: Optional[Any] = None) -> Dict[str, Dict[str, Any]]:
    """
    doc = stanza_pipeline(text)

    Returns one dict per sentence.
    """
    rows = {}

    for i, sent in enumerate(doc.sentences):
        tree = sent.constituency

        row = {
            # "sentence_text": sent.text,
            "n_words": count_non_punct_words(tree),
            "left_embedding_depth": left_embedding_depth(tree),
            "center_embedding_depth": center_embedding_depth(tree),
        }

        yngve = yngve_scores(tree)
        row["yngve_sum"] = yngve["sum"]
        row["yngve_mean"] = yngve["mean"]
        row["yngve_max"] = yngve["max"]
        #row["yngve_per_word"] = yngve["per_word"]

        frazier = frazier_scores(tree)
        row["frazier_sum"] = frazier["sum"]
        row["frazier_mean"] = frazier["mean"]
        row["frazier_max"] = frazier["max"]
        #row["frazier_per_word"] = frazier["per_word"]

        if surprisal_model is not None:
            s = surprisal_model.sentence_surprisal_summary(sent.text)
            row["surprisal_sum"] = s["sum"]
            row["surprisal_mean"] = s["mean"]
            row["surprisal_max"] = s["max"]
            row["surprisal_per_token"] = s["per_token"]

        rows[f"Sentence {i+1}"] = row

    return rows

In [ ]:
#nlp = ba.BatchalignPipeline.new("morphosyntax", lang="eng", num_speakers=1)
stanza_pipeline = stanza.Pipeline(lang="en", processors="tokenize,mwt,pos,lemma,depparse,constituency")

2026-03-10 11:23:14 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


2026-03-10 11:23:15 INFO: Downloaded file to C:\Users\great\AppData\Local\StanfordNLP\stanza\Cache\1.11.0\resources\resources.json
2026-03-10 11:23:16 INFO: Loading these models for language: en (English):
| Processor    | Package             |
--------------------------------------
| tokenize     | combined            |
| mwt          | combined            |
| pos          | combined_charlm     |
| lemma        | combined_nocharlm   |
| constituency | ptb3-revised_charlm |
| depparse     | combined_charlm     |

2026-03-10 11:23:16 INFO: Using device: cpu
2026-03-10 11:23:16 INFO: Loading: tokenize
2026-03-10 11:23:23 INFO: Loading: mwt
2026-03-10 11:23:23 INFO: Loading: pos
2026-03-10 11:23:25 INFO: Loading: lemma
2026-03-10 11:23:26 INFO: Loading: constituency
2026-03-10 11:23:26 INFO: Loading: depparse
2026-03-10 11:23:27 INFO: Done loading processors!


In [38]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
features = {}
i = 0
for abstract in tqdm(test_news_df[:100]["abstract"]):
    #features[abstract_id] = {} #type: ignore
    abstract_id = test_news_df.iloc[i, 0]
    stanza_doc = stanza_pipeline(abstract)
    features[abstract_id] = analyze_doc(stanza_doc) #type: ignore
    features[abstract_id]["Number of Sentences"] = len(stanza_doc.sentences) # type: ignore Also corresponds to the number of clauses
    features[abstract_id]["Mean Sentence Length (characters)"] = sum(len(s.tokens) for s in stanza_doc.sentences) / len(stanza_doc.sentences) #type: ignore
    features[abstract_id]["Density of Clauses"] = clause_density(stanza_doc) #type: ignore
    features[abstract_id]["Mean Dependency Length"] = mean_dependency_length(stanza_doc) #type: ignore
    #pprint(analyze_doc(stanza_doc)) #type: ignore
    i += 1
pprint(features)

100%|██████████| 100/100 [01:53<00:00,  1.14s/it]

{'N101607': {'Density of Clauses': [3, 2, 1, 1],
             'Mean Dependency Length': [4.185185185185185,
                                        2.7,
                                        2.736842105263158,
                                        2.125],
             'Mean Sentence Length (characters)': 19.5,
             'Number of Sentences': 4,
             'Sentence 1': {'center_embedding_depth': 0,
                            'frazier_max': 5.0,
                            'frazier_mean': 1.7083333333333333,
                            'frazier_sum': 41.0,
                            'left_embedding_depth': 3,
                            'n_words': 24,
                            'yngve_max': 6,
                            'yngve_mean': 3.2083333333333335,
                            'yngve_sum': 77},
             'Sentence 2': {'center_embedding_depth': 0,
                            'frazier_max': 5.5,
                            'frazier_mean': 2.5,
                       

# [Bing News API (Official Documentation)](https://learn.microsoft.com/en-us/previous-versions/bing/search-apis/bing-news-search/reference/query-parameters#category)

## [Bing News API (Unofficial Documentation)](https://www.searchapi.io/docs/bing-news)

Search query (q): I do not know what to do with this on startup

category: Might be useful for similarity searching (MaxClass for Top Stories category)

device: desktop, mobile, or tablet

time_period: last_minute to last_30_days

sort_by: default is relevance or can be set to most_recent

num: 50 for our use

page: can go from 1 to 3 for initial purposes

engine: bing_news

api_key: 

zero_retention: enterprise only feature, can be set to true, useful for high-compliance use cases (internal company use?)

Relevant articles:
[Limit of results?](https://learn.microsoft.com/en-us/answers/questions/2125940/bing-news-api)

'{\n  "error": "Invalid API key."\n}'